# English to Hindi Neural Machine Translation with Encoder and Decoder

## Purpose
This notebook demonstrates a basic English-to-Hindi neural machine translation (NMT) system using a sequence-to-sequence (Seq2Seq) model with an Encoder-Decoder architecture implemented in TensorFlow/Keras. The goal is to translate simple English sentences into their Hindi equivalents.

## 1. Imports
We begin by importing the necessary libraries for numerical operations, data structures, and TensorFlow for building the neural network.

In [ ]:
import tensorflow as tf
import numpy as np
from collections import Counter
import re
from tensorflow.keras.preprocessing.sequence import pad_sequences

## 2. Data Loading and Separation
We define a small dataset of English-Hindi sentence pairs and then separate them into two lists: one for English sentences and one for Hindi sentences.

In [ ]:
data = [ ("i am happy","मैं खुश हूँ"),
         ("You are sad","आप दुखी हैं"),
         ("she is tired", "वह थक गया है"),
         ("we are hungry","हम भूखें है"),
         ("they are busy","वे व्यस्त हैं"),
         ("i am cold","मुझे ठंड लग रही है"),
         ("you are late","तुम देरी से आए हो"),
         ("she is happy", "वह खुश है"),
         ("we are ready","हम तैयार हैं") ]

In [ ]:
# Separate English and Hindi sentences
eng_sen = []
hin_sen = []

for pair in data:
  eng_sen.append(pair[0])
  hin_sen.append(pair[1])

print("English Sentences:", eng_sen)
print("Hindi Sentences:", hin_sen)

English Sentences: ['i am happy', 'You are sad', 'she is tired', 'we are hungry', 'they are busy', 'i am cold', 'you are late', 'she is happy', 'we are ready']
Hindi Sentences: ['मैं खुश हूँ', 'आप दुखी हैं', 'वह थक गया है', 'हम भूखें है', 'वे व्यस्त हैं', 'मुझे ठंड लग रही है', 'तुम देरी से आए हो', 'वह खुश है', 'हम तैयार हैं']


## 3. Tokenization
Next, we tokenize each sentence (split it into words). This is a crucial step for preparing text data for machine learning models.

In [ ]:
# Tokenizing the sentences
eng_token = []
hin_token = []

for i in eng_sen:
  eng_token.append(i.split())

for i in hin_sen:
  hin_token.append(i.split())

print("English Tokens:", eng_token)
print("Hindi Tokens:", hin_token)

English Tokens: [['i', 'am', 'happy'], ['You', 'are', 'sad'], ['she', 'is', 'tired'], ['we', 'are', 'hungry'], ['they', 'are', 'busy'], ['i', 'am', 'cold'], ['you', 'are', 'late'], ['she', 'is', 'happy'], ['we', 'are', 'ready']]
Hindi Tokens: [['मैं', 'खुश', 'हूँ'], ['आप', 'दुखी', 'हैं'], ['वह', 'थक', 'गया', 'है'], ['हम', 'भूखें', 'है'], ['वे', 'व्यस्त', 'हैं'], ['मुझे', 'ठंड', 'लग', 'रही', 'है'], ['तुम', 'देरी', 'से', 'आए', 'हो'], ['वह', 'खुश', 'है'], ['हम', 'तैयार', 'हैं']]


## 4. Vocabulary Creation and Word-to-Number Mapping
We create a vocabulary of all unique words for both English and Hindi. Then, we map each unique word to a unique integer ID. This numerical representation is required for neural networks.

We first build raw lists of all words, then use `Counter` to get word frequencies (though we only need the unique words here), and finally convert them to unique lists and create `word2num` dictionaries.

In [ ]:
# VOCAB - list all the unique words(tokens)
eng_vocab_list = []
hin_vocab_list = []

for i in eng_token:
  for j in i:
    eng_vocab_list.append(j)

for i in hin_token:
  for j in i:
    hin_vocab_list.append(j)

print("Raw English Vocabulary List:", eng_vocab_list[:10], "...")

Raw English Vocabulary List: ['i', 'am', 'happy', 'You', 'are', 'sad', 'she', 'is', 'tired', 'we'] ...


In [ ]:
# Use Counter (optional, but shows frequencies)
eng_vocab_counter = Counter(eng_vocab_list)
hin_vocab_counter = Counter(hin_vocab_list)

print("English Vocabulary with Counts:", eng_vocab_counter)

English Vocabulary with Counts: Counter({'are': 5, 'i': 2, 'am': 2, 'happy': 2, 'she': 2, 'is': 2, 'we': 2, 'You': 1, 'sad': 1, 'tired': 1, 'hungry': 1, 'they': 1, 'busy': 1, 'cold': 1, 'you': 1, 'late': 1, 'ready': 1})


In [ ]:
# Get unique words by converting to set and back to list
eng_vocab = sorted(list(set(eng_vocab_list)))
hin_vocab = sorted(list(set(hin_vocab_list)))

print("Unique English Vocabulary:", eng_vocab)

Unique English Vocabulary: ['You', 'am', 'are', 'busy', 'cold', 'happy', 'hungry', 'i', 'is', 'late', 'ready', 'sad', 'she', 'they', 'tired', 'we', 'you']


In [ ]:
# Convert words into numbers (word-to-index mapping)
eng_word2num = {}
hin_word2num = {}

for i,j in enumerate(eng_vocab):
  eng_word2num[j] = i + 1 # Start indexing from 1, 0 often reserved for padding

for i,j in enumerate(hin_vocab):
  hin_word2num[j] = i + 1 # Start indexing from 1

print("English Word to Number Mapping:", eng_word2num)

English Word to Number Mapping: {'You': 1, 'am': 2, 'are': 3, 'busy': 4, 'cold': 5, 'happy': 6, 'hungry': 7, 'i': 8, 'is': 9, 'late': 10, 'ready': 11, 'sad': 12, 'she': 13, 'they': 14, 'tired': 15, 'we': 16, 'you': 17}


## 5. Sequence Preparation and Padding
Sentences are converted into sequences of numerical IDs. For the Hindi (target) sentences, we add special `<start>` and `<end>` tokens to mark the beginning and end of a sequence, which helps the decoder know when to start and stop generating. Finally, all sequences are padded to a uniform length to allow batch processing.

In [ ]:
# Convert tokenized sentences into numerical sequences
eng_sequence =[]
hin_sequence =[]

# English sequences
for i in eng_token:
  temp = []
  for j in i:
    temp.append(eng_word2num[j])
  eng_sequence.append(temp)

# Hindi sequences
for i in hin_token:
  temp = []
  for j in i:
    temp.append(hin_word2num[j])
  hin_sequence.append(temp)

print("English Number Sequences:", eng_sequence)
print("Hindi Number Sequences:", hin_sequence)

English Number Sequences: [[8, 2, 6], [1, 3, 12], [13, 9, 15], [16, 3, 7], [14, 3, 4], [8, 2, 5], [17, 3, 10], [13, 9, 6], [16, 3, 11]]
Hindi Number Sequences: [[13, 3, 21], [2, 9, 23], [16, 8, 4, 22], [20, 11, 22], [17, 18, 23], [12, 5, 15, 14, 22], [6, 10, 19, 1, 24], [16, 3, 22], [20, 7, 23]]


In [ ]:
# Add <start> and <end> tokens to Hindi vocabulary and sequences
hin_word2num['<start>'] = len(hin_word2num) + 1
hin_word2num['<end>'] = len(hin_word2num) + 1

n_hin_sequence = []
for i in hin_sequence:
  start = [hin_word2num['<start>']]
  end = [hin_word2num['<end>']]
  temp = start + i + end
  n_hin_sequence.append(temp)

hin_sequence = n_hin_sequence

print("Hindi sequences with <start> and <end>:", hin_sequence)

Hindi sequences with <start> and <end>: [[25, 13, 3, 21, 26], [25, 2, 9, 23, 26], [25, 16, 8, 4, 22, 26], [25, 20, 11, 22, 26], [25, 17, 18, 23, 26], [25, 12, 5, 15, 14, 22, 26], [25, 6, 10, 19, 1, 24, 26], [25, 16, 3, 22, 26], [25, 20, 7, 23, 26]]


In [ ]:
# Pad sequences to ensure uniform length for batch processing
# 'post' padding means zeros are added to the end
eng_sequence = pad_sequences(eng_sequence, padding='post')
hin_sequence = pad_sequences(hin_sequence, padding='post')

print("Padded English sequences:\n", eng_sequence)
print("Padded Hindi sequences:\n", hin_sequence)

Padded English sequences:
 [[ 8  2  6]
 [ 1  3 12]
 [13  9 15]
 [16  3  7]
 [14  3  4]
 [ 8  2  5]
 [17  3 10]
 [13  9  6]
 [16  3 11]]
Padded Hindi sequences:
 [[25 13  3 21 26  0  0]
 [25  2  9 23 26  0  0]
 [25 16  8  4 22 26  0]
 [25 20 11 22 26  0  0]
 [25 17 18 23 26  0  0]
 [25 12  5 15 14 22 26]
 [25  6 10 19  1 24 26]
 [25 16  3 22 26  0  0]
 [25 20  7 23 26  0  0]]


## 6. Model Architecture and Compilation
We define the Seq2Seq model using TensorFlow's Keras API. It consists of an Encoder (LSTM) that processes the input English sentence and a Decoder (LSTM) that generates the Hindi translation, conditioned on the encoder's output.

In [ ]:
# Define sizes for vocabulary, embedding, and LSTM units
eng_vocab_size = len(eng_word2num) + 1
hin_vocab_size = len(hin_word2num) + 1

embedding_dim = 64
units = 128

print(f"English Vocabulary Size: {eng_vocab_size}")
print(f"Hindi Vocabulary Size: {hin_vocab_size}")

English Vocabulary Size: 18
Hindi Vocabulary Size: 27


In [ ]:
# Encoder
# Takes English word sequences as input
encoder_inputs = tf.keras.Input(shape=(None,))
encoder_embedding = tf.keras.layers.Embedding(eng_vocab_size, embedding_dim)(encoder_inputs)
encoder_lstm = tf.keras.layers.LSTM(units, return_state=True) # return_state to pass context to decoder
encoder_output, state_h, state_c = encoder_lstm(encoder_embedding)
# state_h and state_c are the context vectors (hidden state and cell state) from the encoder

In [ ]:
# Decoder
# Takes Hindi word sequences as input (with <start> tokens)
decoder_inputs = tf.keras.Input(shape=(None,))
decoder_embedding = tf.keras.layers.Embedding(hin_vocab_size, embedding_dim)(decoder_inputs)

decoder_lstm = tf.keras.layers.LSTM(units, return_sequences=True, return_state=True) # return_sequences to output a sequence, return_state for inference

decoder_outputs, _, _ = decoder_lstm(
    decoder_embedding, initial_state=[state_h, state_c] # Initialize decoder with encoder's context
)

decoder_dense = tf.keras.layers.Dense(hin_vocab_size, activation='softmax') # Output layer predicts next Hindi word
decoder_outputs = decoder_dense(decoder_outputs)

# Combine Encoder and Decoder into a single model
model = tf.keras.Model([encoder_inputs, decoder_inputs], decoder_outputs)

model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_1       │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding           │ (None, None, 64)  │      1,152 │ input_layer[0][0] │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_1         │ (None, None, 64)  │      1,728 │ input_layer_1[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm (LSTM)         │ [(None, 128),     │     98,816 │ embedding[0][0]   │
│                     │ (None, 128),      │            │                   │
│                     │ (None, 128)]      │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_1 (LSTM)       │ [(None, None,     │     98,816 │ embedding_1[0][0… │
│                     │ 128), (None,      │            │ lstm[0][1],       │
│                     │ 128), (None,      │            │ lstm[0][2]        │
│                     │ 128)]             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, None, 27)  │      3,483 │ lstm_1[0][0]      │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 203,995 (796.86 KB)

 Trainable params: 203,995 (796.86 KB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
# Compile the model
# Optimizer: 'adam' for efficient gradient descent
# Loss: 'sparse_categorical_crossentropy' suitable for integer-encoded target labels
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy')

## 7. Model Training
We prepare the decoder's input and output data. The decoder input is the Hindi sequence shifted by one position (starting with `<start>`), and the decoder output is the actual Hindi sequence (ending with `<end>`). The model is then trained using these prepared sequences.

In [ ]:
# Prepare decoder input and output data for training
# decoder_input_data: All Hindi sequences except the last token (e.g., <start> word1 word2 ...)
decoder_input_data = hin_sequence[:, :-1]
# decoder_output_data: All Hindi sequences except the first token (e.g., word1 word2 ... <end>)
decoder_output_data = hin_sequence[:, 1:]

print("Decoder Input Data Shape:", decoder_input_data.shape)
print("Decoder Output Data Shape:", decoder_output_data.shape)

Decoder Input Data Shape: (9, 6)
Decoder Output Data Shape: (9, 6)


In [ ]:
# Train the model
# We pass English sequences to the encoder and shifted Hindi sequences to the decoder
model.fit(
    [eng_sequence, decoder_input_data],
    decoder_output_data,
    epochs=200 # Number of training iterations
)

Epoch 1/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 4s 4s/step - loss: 3.2987
Epoch 2/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 61ms/step - loss: 3.2891
Epoch 3/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 58ms/step - loss: 3.2794
Epoch 4/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 56ms/step - loss: 3.2690
Epoch 5/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 55ms/step - loss: 3.2578
Epoch 6/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 59ms/step - loss: 3.2452
Epoch 7/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 57ms/step - loss: 3.2308
Epoch 8/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 57ms/step - loss: 3.2140
Epoch 9/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 57ms/step - loss: 3.1942
Epoch 10/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 57ms/step - loss: 3.1705
Epoch 11/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 56ms/step - loss: 3.1418
Epoch 12/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 57ms/step - loss: 3.1069
Epoch 13/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 57ms/step - loss: 3.0640
Epoch 14/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 55ms/step - loss: 3.0111
Epoch 15/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 61ms/step - loss: 2.9456
Epoch 16/200
1/1 ━━━━

## 8. Translation Function and Demonstration
After training, we define a function to perform step-by-step translation. This function takes an English sentence, tokenizes it, feeds it to the trained model, and generates the Hindi translation word by word until an `<end>` token is predicted or a maximum length is reached.

In [ ]:
def translate_step_by_step(sentence):
    print("\nInput Sentence:", sentence)

    # Step 1: Convert input English sentence to numbers
    # Handle words not in vocabulary by assigning 0 or a special unknown token if implemented
    test_seq = [eng_word2num.get(word, 0) for word in sentence.split()]
    if 0 in test_seq: # Check if any unknown words were found
        print("Warning: Input sentence contains unknown words. Translation might be inaccurate.")
    test_seq = np.array(test_seq).reshape(1, -1) # Reshape for model input

    print("Converted to numbers:", test_seq)

    # Step 2: Initialize decoder input with the <start> token
    start_token = hin_word2num['<start>']
    end_token = hin_word2num['<end>']

    decoder_input = np.array([[start_token]])

    print("Starting decoder with <start> token.\n")

    output = []
    idx2word = {v:k for k,v in hin_word2num.items()} # Create number-to-word mapping for output

    for step in range(15): # Max 15 steps to prevent infinite loop for longer sentences
        # Predict the next word
        pred = model.predict([test_seq, decoder_input], verbose=0)
        next_word_idx = np.argmax(pred[0, -1, :]) # Get index of the word with highest probability

        word = idx2word.get(next_word_idx, "?") # Convert index back to word

        print(f"Step {step+1}: Predicted word → {word}")

        if next_word_idx == end_token:
            print("Reached <end> token. STOP.")
            break

        output.append(word)

        # Add the predicted word to the decoder input for the next step
        decoder_input = np.append(decoder_input, [[next_word_idx]], axis=1)

    print("\nFinal Translation:", " ".join(output))


In [ ]:
# Test the translation function with an example sentence
translate_step_by_step('they are busy')


Input Sentence: they are busy
Converted to numbers: [[14  3  4]]
Starting decoder with <start> token.

Step 1: Predicted word → वे
Step 2: Predicted word → व्यस्त
Step 3: Predicted word → हैं
Step 4: Predicted word → <end>
Reached <end> token. STOP.

Final Translation: वे व्यस्त हैं


In [ ]:
# You can try other sentences from your dataset:
translate_step_by_step('i am happy')
translate_step_by_step('she is tired')
translate_step_by_step('we are hungry')



Input Sentence: i am happy
Converted to numbers: [[8 2 6]]
Starting decoder with <start> token.

Step 1: Predicted word → मैं
Step 2: Predicted word → खुश
Step 3: Predicted word → हूँ
Step 4: Predicted word → <end>
Reached <end> token. STOP.

Final Translation: मैं खुश हूँ

Input Sentence: she is tired
Converted to numbers: [[13  9 15]]
Starting decoder with <start> token.

Step 1: Predicted word → वह
Step 2: Predicted word → थक
Step 3: Predicted word → गया
Step 4: Predicted word → है
Step 5: Predicted word → <end>
Reached <end> token. STOP.

Final Translation: वह थक गया है

Input Sentence: we are hungry
Converted to numbers: [[16  3  7]]
Starting decoder with <start> token.

Step 1: Predicted word → हम
Step 2: Predicted word → भूखें
Step 3: Predicted word → है
Step 4: Predicted word → <end>
Reached <end> token. STOP.

Final Translation: हम भूखें है
